# Quadrotor Dynamics on SE(3)

Derives the equations of motion for a quadrotor UAV modeled as a rigid body
with position $x \in \mathbb{R}^3$ and attitude $R \in SO(3)$.

**Configuration manifold**: $\mathbb{R}^3 \times SO(3)$ (SE(3) as a product manifold)

**Inputs**: Thrust scalar $f$ along body $e_3$ axis, moment vector $M$ in body frame

**Expected EOM** (Lee et al. 2010, Geometric Tracking Control):
$$m\ddot{x} + mge_3 = fRe_3$$
$$J\dot{\Omega} + \Omega \times J\Omega = M$$

In [33]:
# Install from GitHub (for Colab); uncomment if not installed locally
# !pip install -q git+https://github.com/vkotaru/pygeomech.git@geomech

from geomech import (
    SO3, Scalar, Vector, Matrix, Dot,
    SystemVariables, TimeDerivative, Variation,
    compute_eom, to_standard_form, getScalars,
    to_latex, display_latex, display_eom, display_standard_form,
)
from geomech.core.operations.multiplication import MVMul, SVMul
from geomech.utils.printing import print_tree
from IPython.display import Math

In [34]:
# Parameters
m = Scalar('m', attr=['Constant'])     # mass
g = Scalar('g', attr=['Constant'])     # gravity
J = Matrix('J', attr=['Constant', 'SymmetricMatrix'])  # body inertia
e3 = Vector('e3', attr=['Constant'])   # gravity direction
half = Scalar('0.5', value=0.5, attr=['Constant'])

# Configuration variables: x ∈ R³, R ∈ SO(3)
x = Vector('x')
R = SO3('R')
Om = R.get_tangent_vector()     # body angular velocity Ω
eta = R.get_variation_vector()  # variation vector η

# Inputs
f_thrust = Scalar('f')   # thrust magnitude
M_torque = Vector('M')   # body-frame torque

# Thrust force in world frame
thrust_force = SVMul(MVMul(R, e3), f_thrust)

print('States:  x ∈ R³,  R ∈ SO(3)')
print('Ω =', Om)
print('η =', eta)
display(Math(r'F_{\text{thrust}} = ' + to_latex(thrust_force)))

States:  x ∈ R³,  R ∈ SO(3)
Ω = \Omega_{R}
η = \eta_{R}


<IPython.core.display.Math object>

## Lagrangian

$$L = \frac{1}{2} m \|\dot{x}\|^2 + \frac{1}{2} \Omega^T J \Omega - m g \, x \cdot e_3$$

In [35]:
v = x.t_diff()
KE = m * Dot(v, v) * half + Dot(Om, J * Om) * half
PE = m * g * Dot(x, e3)
L = KE - PE

display(Math(r'L = ' + to_latex(L)))

<IPython.core.display.Math object>

## Infinitesimal work

$$\delta W = \delta x \cdot (f R e_3) + \eta \cdot M$$

In [36]:
dW = Dot(x.delta(), thrust_force) + Dot(eta, M_torque)
display(Math(r'\delta W = ' + to_latex(dW)))

<IPython.core.display.Math object>

## Equations of motion

In [37]:
variables = SystemVariables(vectors=[x], matrices=[R])
eom = compute_eom(L, dW, variables)

display_eom(eom)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Standard form

$$M(q) \ddot{q} + f(q, \dot{q}) + G(q) u = 0$$

In [38]:
sf = to_standard_form(eom, variables, [thrust_force, M_torque])
display_standard_form(sf)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Verify

### Translational: $m\ddot{x} + mge_3 = fRe_3$
### Rotational: $J\dot{\Omega} + \Omega \times J\Omega = M$

In [39]:
# Translational
key_trans = str(Variation(x))
eq_trans = sf[key_trans]
ddx = str(TimeDerivative(TimeDerivative(x)))

display(Math(r'\textbf{Translational:}'))
display(Math(r'M[\ddot{x}] = ' + to_latex(eq_trans.M[ddx])))
display(Math(r'f = ' + to_latex(eq_trans.f)))
for k, v in eq_trans.G.items():
    display(Math(f'G[{to_latex(thrust_force)}] = ' + to_latex(v)))

print()

# Rotational
key_rot = str(eta)
eq_rot = sf[key_rot]
ddOm = str(TimeDerivative(Om))

display(Math(r'\textbf{Rotational:}'))
display(Math(r'M[\dot{\Omega}] = ' + to_latex(eq_rot.M[ddOm])))
display(Math(r'f = ' + to_latex(eq_rot.f)))
display(Math(r'G[M] = ' + to_latex(eq_rot.G['M'])))

print()
print('Translational: m*x_ddot + m*g*e3 = f*R*e3  ✓')
print('Rotational:    J*Om_dot + Om×J*Om = M       ✓')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


Translational: m*x_ddot + m*g*e3 = f*R*e3  ✓
Rotational:    J*Om_dot + Om×J*Om = M       ✓
